In [28]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import os

# Check your drive structure
for item in os.listdir('/content/drive/MyDrive'):
    print(item)

picture white background (1).jpg
picture white background.jpg
Classroom
Afroz (diagnostic test).gdoc
Summary (diagnostic test 2).gdoc
3. Effective Presentation Skills.gdoc
5. drinks.gsheet
VID-20231008-WA0043.mp4
Afroz Talha - Worksheet.docx
Afroz Talha - Worksheet.gdoc
23i-2539_AI_A (1).zip
232539 (1).zip
23i-2539_Afroz_Talha.docx
Word Parts .gdoc
Pecha Kucha presentation.pptx
Final_Draft[1].docx
Workshop_via_Human_Brain[1].docx
IMG20231110101225.jpg
232539.zip
snake
PF Lab 12.gdoc
23I-2539_Afroz_Talha_BS-AI-(A) 1.docx.pdf
23I-2539_Afroz_Talha_BS-AI-(A)1.docx.pdf
Afroz Talha _23I-2539_A
ict
Afroz Talha_LAB TASK 5_BS-AI-A.gsheet
Afroz_Talha_23I-2539_homework1 (5).pdf
Afroz_Talha_23I-2539_homework1 (4).pdf
Afroz_Talha_23I-2539_homework1 (3).pdf
Afroz_Talha_23I-2539_homework1 (2).pdf
Afroz_Talha_23I-2539_homework1 (1).pdf
Afroz_Talha_23I-2539_homework1.pdf
23i-2539_BS-AI-A_Afroz_Talha (2).pdf
23i-2539_BS-AI-A_Afroz_Talha (1).pdf
23i-2539_BS-AI-A_Afroz_Talha.pdf
23i-2539_AfrozTalha.pdf
PF

In [14]:
# Confirm the MIT-BIH files are there
dataset_path = '/content/drive/MyDrive/mit-bih-arrhythmia-database-1.0.0'
files = os.listdir(dataset_path)
print(f"Files found: {len(files)}")
print(files[:10])  # show first 10

Files found: 207
['101.hea', '103.dat', '103.hea', '100.atr', '105.dat', '103.xws', '104.xws', '102.atr', '104.atr', '106.atr']


# ECG Graph Construction Pipeline (Assignment 3 — CNN Extension)

**Pipeline order:**
1. Install dependencies
2. Stage 1 & 2 — ECG signal → waveform PNG images
3. Stage 3 — Prewitt edge detection
4. Label / dataset configuration
5. **CNN encoder setup** ← Assignment 3 addition
6. Stage 4 — Graph construction using CNN node features
7. Save graphs to disk (Trainset_ECG_CNN / Testset_ECG_CNN)

In [5]:
# Install required libraries (run once)
!pip install wfdb
!pip install opencv-python

## Stage 1 & 2 — ECG Signal → Waveform PNG Images

Reads raw `.dat` / `.atr` files from MIT-BIH using `wfdb`.  
For each annotated heartbeat, extracts the signal window, applies PAA compression  
to 100 points, plots it as a 64×64 PNG, inverts colours (white signal on black).  
Saves into `processed_data/images/<CLASS>/`.

In [21]:
import os
import time
import tqdm
import wfdb
import cv2
import json
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from os import listdir
from os.path import isfile, join

# ── CONFIG ────────────────────────────────────────────────────────────────
size             = 64          # output image size (64x64)
size_paa         = 100         # PAA dimension
_range_to_ignore = 10          # samples to trim from beat edges
_directory   = '/content/drive/MyDrive/mit-bih-arrhythmia-database-1.0.0/'
_dataset_dir = '/content/drive/MyDrive/processed_data/images/'
MAX_RECORDS      = 10          # limit to first 10 records for quick run

# MIT-BIH label mappings
labels_json     = '{ "N": "NOR", "L": "LBBB", "R": "RBBB", "A": "APB", "V": "PVC"}'
labels_to_float = '{ "NOR": "0", "LBBB": "1", "RBBB": "2", "APB": "3", "PVC": "4"}'
float_to_labels = '{ "0": "NOR", "1": "LBBB", "2": "RBBB", "3": "APB", "4": "PVC"}'

labels          = json.loads(labels_to_float)
revert_labels   = json.loads(float_to_labels)
original_labels = json.loads(labels_json)


def piecewise_aggregate_approximation(vector, paa_dim: int):
    Y = np.array(vector)
    if Y.shape[0] % paa_dim == 0:
        sectionedArr = np.array_split(Y, paa_dim)
        res = np.array([item.mean() for item in sectionedArr])
    else:
        value_space  = np.arange(0, Y.shape[0] * paa_dim)
        output_index = value_space // Y.shape[0]
        input_index  = value_space // paa_dim
        uniques, nUniques = np.unique(output_index, return_counts=True)
        res = [Y[indices].sum() / Y.shape[0]
               for indices in np.split(input_index, nUniques.cumsum())[:-1]]
    return res


def create_img_from_sign_filtered(size=(size, size), size_paa=size_paa):
    os.makedirs(_directory,   exist_ok=True)
    os.makedirs(_dataset_dir, exist_ok=True)

    files = [f[:-4] for f in listdir(_directory)
             if isfile(join(_directory, f)) and f.endswith('.dat')]
    files.sort()
    files = files[:MAX_RECORDS]
    print(f"Processing {len(files)} records: {files}")

    for file in files:
        sig, _ = wfdb.rdsamp(_directory + file)
        ann    = wfdb.rdann(_directory + file, extension='atr')

        for i in tqdm.tqdm(range(1, len(ann.sample) - 1),
                           desc=f'Record {file}', ncols=80):
            if ann.symbol[i] not in original_labels:
                continue

            note   = ann.symbol[i]
            label  = original_labels[note]
            start  = ann.sample[i - 1] + _range_to_ignore
            end    = ann.sample[i + 1] - _range_to_ignore
            signal = [sig[j][0] for j in range(start, end)]
            paa    = piecewise_aggregate_approximation(signal, size_paa)

            fig = plt.figure(frameon=False)
            plt.plot(list(range(len(paa))), paa)
            plt.xticks([]), plt.yticks([])
            for spine in plt.gca().spines.values():
                spine.set_visible(False)

            dir_path = f'{_dataset_dir}/{label}'
            os.makedirs(dir_path, exist_ok=True)
            filename = f'{dir_path}/{label}_{file[-3:]}_{start}_{end}_0.png'

            fig.savefig(filename, bbox_inches='tight')
            im_gray = cv2.imread(filename, cv2.IMREAD_GRAYSCALE)
            im_gray = cv2.resize(im_gray, size, interpolation=cv2.INTER_LANCZOS4)
            im_gray = np.invert(im_gray)
            cv2.imwrite(filename, im_gray)
            plt.cla(); plt.clf(); plt.close('all')


start_time = time.time()
create_img_from_sign_filtered()
print(f"Stage 1&2 done — {(time.time()-start_time)/60:.2f} min")

Processing 10 records: ['100', '101', '102', '103', '104', '105', '106', '107', '108', '109']


Record 109: 100%|███████████████████████████| 2533/2533 [04:46<00:00,  8.83it/s]

Stage 1&2 done — 29.55 min


## Stage 3 — Prewitt Edge Detection

Applies a Prewitt filter to each waveform image.  
Bright pixels (the ECG line) become strong edges.  
Saves edge-filtered images to `processed_data/edge_filtered/`.

In [22]:
import glob

img_size   = 64
sourcedir = "/content/drive/MyDrive/processed_data/images"
destdir   = "/content/drive/MyDrive/processed_data/edge_filtered"
os.makedirs(destdir, exist_ok=True)


def prewitt_edge(filename):
    img        = cv2.imread(filename, cv2.IMREAD_GRAYSCALE)
    img        = cv2.resize(img, (img_size, img_size))
    kernel     = np.array([[-1, 0, 1], [-1, 0, 1], [-1, 0, 1]], dtype=np.float32)
    h_grad     = cv2.filter2D(img, -1, kernel)
    v_grad     = cv2.filter2D(img, -1, kernel.T)
    magnitude  = np.sqrt(h_grad**2 + v_grad**2).astype(np.float32)
    magnitude  = cv2.normalize(magnitude, None, 0, 255, cv2.NORM_MINMAX)
    return np.uint8(magnitude)


def run_edge_detection(sourcedir, destdir):
    for subdir, _, files in os.walk(sourcedir):
        rel_path = os.path.relpath(subdir, sourcedir)
        if rel_path == '.':
            continue
        out_dir = os.path.join(destdir, rel_path)
        os.makedirs(out_dir, exist_ok=True)
        for file in files:
            full_path = os.path.join(subdir, file)
            edge_img  = prewitt_edge(full_path)
            cv2.imwrite(os.path.join(out_dir, file), edge_img)
    print("Stage 3 done — edge images saved to:", destdir)


start = time.time()
run_edge_detection(sourcedir, destdir)
print(f"Edge detection done — {(time.time()-start)/60:.2f} min")

Stage 3 done — edge images saved to: /content/drive/MyDrive/processed_data/edge_filtered
Edge detection done — 11.57 min


## Dataset Configuration

Switch `DATASET` between `'mitbih'` and `'ptbxl'` here.  
Everything downstream uses `labels`, `revert_labels`, `original_labels`.

In [23]:
# ── DATASET SWITCH — change this to 'ptbxl' when running PTB-XL ──────────
DATASET = 'mitbih'
# ─────────────────────────────────────────────────────────────────────────

if DATASET == 'mitbih':
    labels_json     = '{ "N": "NOR", "L": "LBBB", "R": "RBBB", "A": "APB", "V": "PVC"}'
    labels_to_float = '{ "NOR": "0", "LBBB": "1", "RBBB": "2", "APB": "3", "PVC": "4"}'
    float_to_labels = '{ "0": "NOR", "1": "LBBB", "2": "RBBB", "3": "APB", "4": "PVC"}'
elif DATASET == 'ptbxl':
    labels_json     = '{ "NORM": "NORM", "MI": "MI", "STTC": "STTC", "CD": "CD", "HYP": "HYP" }'
    labels_to_float = '{ "NORM": "0", "MI": "1", "STTC": "2", "CD": "3", "HYP": "4" }'
    float_to_labels = '{ "0": "NORM", "1": "MI", "2": "STTC", "3": "CD", "4": "HYP" }'
else:
    raise ValueError(f"Unknown DATASET: {DATASET!r}")

labels          = json.loads(labels_to_float)
revert_labels   = json.loads(float_to_labels)
original_labels = json.loads(labels_json)
print(f"Dataset: {DATASET} | Classes: {list(revert_labels.values())}")

Dataset: mitbih | Classes: ['NOR', 'LBBB', 'RBBB', 'APB', 'PVC']


## CNN Encoder — Assignment 3 Addition

Instead of using raw pixel brightness (1 number) as the node feature,  
we run a small CNN on the full 64×64 edge image first.  
This produces a **32-dimensional feature vector for every pixel position**.  
When a pixel becomes a graph node in Stage 4, it gets that 32-dim vector  
instead of a single scalar — giving the GNN much richer information.

**Architecture:**
```
Input: (1, 64, 64) grayscale edge image
  → Conv2d(1→16, 3×3, padding=1) + ReLU   # spatial stays 64×64
  → Conv2d(16→32, 3×3, padding=1) + ReLU  # spatial stays 64×64
Output: (32, 64, 64) feature map
```
No pooling — spatial resolution must stay 64×64 so pixel (i,j)  
maps exactly to feature vector at position (i,j).

In [24]:
import torch
import torch.nn as nn

class ECGCNNEncoder(nn.Module):
    """
    Small CNN encoder: 64x64 edge image → (32, 64, 64) feature map.
    No pooling so spatial resolution is preserved — pixel (i,j) maps
    exactly to the 32-dim feature vector at position (i,j).
    """
    def __init__(self, out_channels=32):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),   # 1→16 channels
            nn.ReLU(inplace=True),
            nn.Conv2d(16, out_channels, kernel_size=3, padding=1),  # 16→32
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.encoder(x)   # (B, 32, 64, 64)


CNN_FEAT_DIM = 32
cnn_encoder  = ECGCNNEncoder(out_channels=CNN_FEAT_DIM)
cnn_encoder.eval()   # inference only during preprocessing
print(f"CNN encoder ready — output channels: {CNN_FEAT_DIM}")


def extract_cnn_features(img_path):
    """
    Loads a 64x64 edge image and returns a (64, 64, 32) numpy array.
    Each pixel position (i,j) holds a 32-dim CNN feature vector.
    """
    img        = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    img        = cv2.resize(img, (64, 64))
    img_tensor = torch.from_numpy(img).float() / 255.0
    img_tensor = img_tensor.unsqueeze(0).unsqueeze(0)   # (1, 1, 64, 64)
    with torch.no_grad():
        feat_map = cnn_encoder(img_tensor)               # (1, 32, 64, 64)
    return feat_map.squeeze(0).permute(1, 2, 0).numpy() # (64, 64, 32)

CNN encoder ready — output channels: 32


## Stage 4 — Graph Construction with CNN Node Features

For each edge-filtered image:
1. Run CNN encoder → get `(64, 64, 32)` feature map
2. Every pixel with brightness ≥ 128 becomes a **graph node**
3. That node's feature = the **32-dim CNN vector** at its (i,j) position
4. Edges connect all adjacent bright pixels (8-connectivity)

This replaces the original `generate_graphs()` which used a single  
raw pixel brightness scalar as the node feature.

In [25]:
import pandas as pd

# ── Global state ────────────────────────────────────────────────────────
edges           = []
attrs           = []
graph_id        = 1
node_id         = 1
graph_indicator = []
node_labels     = []
graph_labels    = []


def generate_graphs_cnn(filename, node_label, revert_labels):
    """
    Builds a graph from one edge-filtered image using CNN node features.
    Node feature = 32-dim CNN vector (replaces raw pixel scalar).
    """
    global node_id, edges, attrs, graph_id, node_labels, graph_indicator

    img           = cv2.imread(filename)
    dim1, dim2, _ = img.shape

    # ── CNN runs here — before any node/edge is created ──────────────────
    feat_map = extract_cnn_features(filename)  # (64, 64, 32)

    nodes  = np.full((dim1, dim2), -1)
    cnt    = 0
    attrs1 = []

    # ── Create nodes — bright pixels only ────────────────────────────────
    for i in range(dim1):
        for j in range(dim2):
            b, _, _ = img[i][j]
            if b >= 128:
                nodes[i][j] = node_id
                attrs1.append(feat_map[i, j, :].tolist())  # 32-dim vector
                graph_indicator.append(graph_id)
                node_labels.append([node_label, revert_labels[node_label]])
                node_id += 1
                cnt     += 1

    # ── Create edges — connect adjacent bright pixels ────────────────────
    for i in range(dim1):
        for j in range(dim2):
            if nodes[i][j] != -1:
                li = max(0, i - 1); ri = min(i + 2, dim1)
                lj = max(0, j - 1); rj = min(j + 2, dim2)
                for i1 in range(li, ri):
                    for j1 in range(lj, rj):
                        if (i1 != i or j1 != j) and nodes[i1][j1] != -1:
                            edges.append([nodes[i][j], nodes[i1][j1]])

    # ── Normalize each of the 32 CNN feature dims independently ──────────
    if cnt > 0:
        arr     = np.array(attrs1)       # (cnt, 32)
        m       = arr.mean(axis=0)
        s       = arr.std(axis=0)
        s[s==0] = 1                      # avoid divide-by-zero
        attrs.extend(((arr - m) / s).tolist())
    if cnt != 0:
        graph_id += 1


def generate_graph_with_labels(dirname, label, revert_labels):
    global graph_labels
    print(f"  Reading: {dirname}")
    for filename in glob.glob(dirname + '/*.png'):
        generate_graphs_cnn(filename, label, revert_labels)
        graph_labels.append([label, revert_labels[label]])


def process_graphs(directory, revert_labels):
    for subdir, _, files in os.walk(directory):
        rel_path = os.path.relpath(subdir, directory)
        if rel_path == '.':
            continue
        generate_graph_with_labels(subdir, labels[rel_path], revert_labels)
    print(f"Done — nodes: {len(node_labels)}  graphs: {len(graph_labels)}")


main_dir = "/content/drive/MyDrive/processed_data/edge_filtered"
start    = time.time()
process_graphs(main_dir, revert_labels)
print(len(node_labels), len(graph_labels), len(edges), len(attrs))
print(f"Stage 4 done — {(time.time()-start)/60:.2f} min")

  Reading: /content/drive/MyDrive/processed_data/edge_filtered/NOR
  Reading: /content/drive/MyDrive/processed_data/edge_filtered/APB
  Reading: /content/drive/MyDrive/processed_data/edge_filtered/PVC
  Reading: /content/drive/MyDrive/processed_data/edge_filtered/LBBB
Done — nodes: 2951453  graphs: 15424
2951453 15424 9700678 2951453
Stage 4 done — 14.64 min


## Build DataFrames

Node attributes now have **32 columns** (one per CNN feature dimension)  
instead of 1 column (raw pixel value) in the original Assignment 2 pipeline.

In [26]:
feat_cols = [f'cnn_{k}' for k in range(CNN_FEAT_DIM)]

df_A              = pd.DataFrame(columns=['node-1', 'node-2'],
                                 data=np.array(edges))
df_node_label     = pd.DataFrame(data=np.array(node_labels),
                                 columns=['label', 'name'])
df_graph_label    = pd.DataFrame(data=np.array(graph_labels),
                                 columns=['label', 'name'])
df_node_attr      = pd.DataFrame(data=np.array(attrs),
                                 columns=feat_cols)
df_graph_indicator = pd.DataFrame(data=np.array(graph_indicator),
                                  columns=['graph-id'])

df_node_label  = df_node_label.drop(['name'],  axis=1)
df_graph_label = df_graph_label.drop(['name'], axis=1)

print("Edges:          ", df_A.shape)
print("Node labels:    ", df_node_label.shape)
print("Graph labels:   ", df_graph_label.shape)
print("Node attributes:", df_node_attr.shape)   # should be (N, 32)
print("Graph indicator:", df_graph_indicator.shape)

Edges:           (9700678, 2)
Node labels:     (2951453, 1)
Graph labels:    (15424, 1)
Node attributes: (2951453, 32)
Graph indicator: (2951453, 1)


## Train / Test Split and Save

Stratified 80/20 split.  
Saved to `Trainset_ECG_CNN` and `Testset_ECG_CNN`  
(separate from the original Assignment 2 folders so both coexist).

In [27]:
from sklearn.model_selection import train_test_split

graph_ids       = df_graph_indicator['graph-id'].unique()
graph_id_labels = df_graph_label['label'].values

train_graphs, test_graphs = train_test_split(
    graph_ids, test_size=0.2, random_state=42, stratify=graph_id_labels
)
print("Train graphs:", len(train_graphs))
print("Test graphs: ", len(test_graphs))


def save_dataframe_to_txt(df, filepath):
    df.to_csv(filepath, header=None, index=None, sep=',', mode='w')


def save_split(graph_subset, split_name, sourcepath):
    path = f"{sourcepath}/{split_name}/raw"
    os.makedirs(path, exist_ok=True)

    mask_nodes          = df_graph_indicator['graph-id'].isin(graph_subset)
    df_node_label_split = df_node_label[mask_nodes]
    df_node_attr_split  = df_node_attr[mask_nodes]
    df_graph_ind_split  = df_graph_indicator[mask_nodes].copy()

    train_node_ids = set(df_graph_indicator[mask_nodes].index + 1)
    df_A_split = df_A[
        df_A['node-1'].isin(train_node_ids) &
        df_A['node-2'].isin(train_node_ids)
    ].copy()

    old_to_new = {old: new+1 for new, old in enumerate(sorted(train_node_ids))}
    df_A_split['node-1'] = df_A_split['node-1'].map(old_to_new)
    df_A_split['node-2'] = df_A_split['node-2'].map(old_to_new)

    old_graph_to_new = {old: new+1 for new, old in enumerate(sorted(graph_subset))}
    df_graph_ind_split['graph-id'] = df_graph_ind_split['graph-id'].map(old_graph_to_new)

    df_graph_label_split = df_graph_label.iloc[graph_subset - 1]

    save_dataframe_to_txt(df_A_split,            f'{path}/{split_name}_A.txt')
    save_dataframe_to_txt(df_graph_ind_split,    f'{path}/{split_name}_graph_indicator.txt')
    save_dataframe_to_txt(df_graph_label_split,  f'{path}/{split_name}_graph_labels.txt')
    save_dataframe_to_txt(df_node_attr_split,    f'{path}/{split_name}_node_attributes.txt')
    save_dataframe_to_txt(df_node_label_split,   f'{path}/{split_name}_node_labels.txt')
    print(f"{split_name} saved — nodes: {len(df_graph_ind_split)}")


base = "/content/drive/MyDrive/processed_data"
save_split(train_graphs, "Trainset_ECG_CNN", base)
save_split(test_graphs,  "Testset_ECG_CNN",  base)
print("=== graph_construct_cnn complete ===")

Train graphs: 12339
Test graphs:  3085
Trainset_ECG_CNN saved — nodes: 2359526
Testset_ECG_CNN saved — nodes: 591927
=== graph_construct_cnn complete ===
